# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstitute_hdf5_from_s3.py` but uses the **local filesystem** for Parquet storage and **SQLite** for metadata. No cloud credentials needed.

**Workflow**
1. Ingest a GMI granule → Parquet partitions on local disk + SQLite metadata (STARE partition at **level 4**)
2. Find intersecting data for a bounding box via STARE SIDs + SQLite (level 4)
3. Load intersecting Parquet partitions from disk
4. Reconstitute an HDF5 file (both S1 and S2 scans) from the level-4 Parquet partitions
5. Compare the reconstituted structure with the original granule
6. Verify SQLite metadata

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
#                       "-q"])

In [2]:
import os
import sqlite3
import h5py
from starepandas.demo_lib import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [3]:
# Parquet store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
import starepandas
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5",
    ),
)

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/reconsitution/gmi_local_reconstituted.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstituted HDF5 (e.g. 3× the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"STARE level: {STARE_LEVEL}")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

Granule    : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
Datasets   : ['GMI_S1', 'GMI_S2']
BBox       : None  (None = full granule)
STARE level: 4
Local root : /tmp/stare_pods_local
Clean first: True


## Step 1 — Ingest granule → local Parquet + SQLite

In [4]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

Removed existing data at /tmp/stare_pods_local


In [5]:
%%time
import time
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=STARE_LEVEL)
print(f"Written {len(local_paths)} scan path(s).")
for p in local_paths:
    print(f"  {p}")

INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5 (granule=1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


Written 2 scan path(s).
  /tmp/stare_pods_local
  /tmp/stare_pods_local
CPU times: user 7.72 s, sys: 465 ms, total: 8.19 s
Wall time: 8.19 s


## Step 2 — Find intersecting data via STARE SIDs

In [6]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all partitions will be loaded (full granule reconstitution)")

intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'])
print(f"Found {len(intersecting)} metadata row(s).")
intersecting[['Dataset', 'grouped_id', 'group_path']]

INFO:starepandas.demo_lib:Loaded all 514 partitions for GMI


No bbox filter — all partitions will be loaded (full granule reconstitution)
Found 514 metadata row(s).


,Dataset,grouped_id,group_path
0,GMI_S2,2094173826727280644,/tmp/stare_pods_local/q32/q322/q3220/q32202/q3...
1,GMI_S2,2118943624677818372,/tmp/stare_pods_local/q32/q322/q3223/q32231/q3...
2,GMI_S2,2163979620951523332,/tmp/stare_pods_local/q33/q330/q3300/q33001/q3...
3,GMI_S2,2193253018529431556,/tmp/stare_pods_local/q33/q330/q3303/q33032/q3...
4,GMI_S2,2096425626540965892,/tmp/stare_pods_local/q32/q322/q3220/q32203/q3...
...,...,...,...
509,GMI_S1,2127950823932559364,/tmp/stare_pods_local/q32/q323/q3230/q32301/q3...
510,GMI_S1,2107684625609392132,/tmp/stare_pods_local/q32/q322/q3222/q32220/q3...
511,GMI_S1,2112188225236762628,/tmp/stare_pods_local/q32/q322/q3222/q32222/q3...
512,GMI_S1,2114440025050447876,/tmp/stare_pods_local/q32/q322/q3222/q32223/q3...


## Step 3 — Load intersecting Parquet partitions from disk

In [7]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions found.")
    data_dict = {}

INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S2


INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S1


GMI_S2: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Quality,incidenceAngle,...,sunLocalTime,incidenceAngleIndex1,incidenceAngleIndex2,incidenceAngleIndex3,incidenceAngleIndex4,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.956139,41.155605,2095643662640840171,2025-01-01 03:43:47.516,256.399994,251.600006,248.050003,255.509995,0,49.57,...,6.415025,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075
1,-60.967239,41.053024,2095644871184156395,2025-01-01 03:43:47.516,256.359985,250.559998,248.229996,254.919998,0,49.57,...,6.408187,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075
2,-60.978867,40.950649,2095644225044829739,2025-01-01 03:43:47.516,255.940002,250.279999,247.250000,254.279999,0,49.57,...,6.401363,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075


GMI_S1: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Tc5,Tc6,...,incidenceAngleIndex5,incidenceAngleIndex6,incidenceAngleIndex7,incidenceAngleIndex8,incidenceAngleIndex9,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.416740,40.916428,2096919498826664619,2025-01-01 03:43:47.516,162.190002,88.510002,183.320007,116.150002,206.300003,209.000000,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075
1,-60.429089,40.802353,2096462762016624139,2025-01-01 03:43:47.516,162.830002,89.080002,183.410004,116.639999,206.479996,209.020004,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075
2,-60.442020,40.688499,2096462641002782731,2025-01-01 03:43:47.516,161.820007,88.800003,183.320007,117.000000,206.729996,209.750000,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [8]:
%%time
import time
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]

recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    granule_name=granule_basename,
)
print(f"Written to: {recon_path}")

INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over bbox=None


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over bbox=None


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/reconsitution/gmi_local_reconstituted.h5


Written to: /tmp/reconsitution/gmi_local_reconstituted.h5
CPU times: user 2.06 s, sys: 385 ms, total: 2.45 s
Wall time: 1.8 s


## Step 5 — Structure comparison: reconstituted vs original

In [9]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_local_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear    

## Step 6 — SQLite metadata verification

In [10]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} partition(s)")

SQLite DB: /tmp/stare_pods_local/metadata.db
  GMI_S1: 263 partition(s)
  GMI_S2: 251 partition(s)
